# Imports

In [ ]:
import os
import json
import torch
import random
import logging
import numpy as np

from torch import nn
from tqdm import tqdm
from pathlib import Path
from torch.optim import AdamW
from transformers import get_scheduler
from torch.utils.data import DataLoader, Dataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers


# Setup

In [ ]:
def set_seed(seed=42):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "Data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "Data"

if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Data directory not found under {PROJECT_ROOT}")

LOG_DIR = PROJECT_ROOT / "Scripts" / "Logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(filename=LOG_DIR / "training.log",
                    filemode="a",

                    format="%(asctime)s - %(levelname)s - %(message)s",
                    level=logging.INFO)

# Settings

In [ ]:
SEQ_LEN = 256
EMBED_DIM = 512
HIDDEN_DIM = 2048
VOCAB_SIZE = 30000

NUM_HEADS = 8
NUM_LAYERS = 12
BATCH_SIZE = 16

EPOCHS = 100
WARMUP_STEPS = 500
FINE_TUNE_LR = 5e-5
LEARNING_RATE = 5e-4
VALIDATION_PATIENCE = 5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nWorking on: {DEVICE}\n")

MODEL_SAVE_DIR = PROJECT_ROOT / "Models" / "literature_v2"
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

TOKENIZER_SAVE_DIR = PROJECT_ROOT / "Models" / "Tokenizer"
TOKENIZER_SAVE_DIR.mkdir(parents=True, exist_ok=True)

TOKENIZER_PATH = TOKENIZER_SAVE_DIR / f"tokenizer_{VOCAB_SIZE}_literature_v2.json"
TOKENIZER_RELATIVE_PATH = str(TOKENIZER_PATH.relative_to(PROJECT_ROOT))

STAGE_ORDER = ["grammar", "simple", "dialogue", "intermediate", "complex"]

STAGE_DATA_CONFIG = {"grammar": {"main": str(DATA_DIR / "grammar.txt"),
                                 "validation": str(DATA_DIR / "grammar_validation.txt")},

                    "simple": {"main": str(DATA_DIR / "simple.txt"),
                               "narrative": str(DATA_DIR / "simple_narrative.txt"),
                               "use_narrative": True,
                               "validation": str(DATA_DIR / "simple_validation.txt")},

                    "dialogue": {"main": str(DATA_DIR / "dialogue.txt"),
                                 "dialogues": str(DATA_DIR / "dialogue_dialogues.txt"),
                                 "use_dialogues": True,
                                 "validation": str(DATA_DIR / "dialogue_validation.txt")},

                    "intermediate": {"main": str(DATA_DIR / "intermediate.txt"),
                                     "dialogues": str(DATA_DIR / "intermediate_dialogues.txt"),
                                     "narrative": str(DATA_DIR / "intermediate_narrative.txt"),
                                     "use_dialogues": True,
                                     "use_narrative": True,
                                     "validation": str(DATA_DIR / "intermediate_validation.txt")},

                    "complex": {"main": str(DATA_DIR / "complex.txt"),
                                "narrative": str(DATA_DIR / "complex_narrative.txt"),
                                "use_narrative": True,
                                "validation": str(DATA_DIR / "complex_validation.txt")}}

SyntaxError: invalid syntax (19846214.py, line 1)

# Loading the Data

In [ ]:
class TextDataset(Dataset):

    def __init__(self, file_path, tokenizer, seq_len=SEQ_LEN):

        with open(file_path, "r", encoding="utf-8") as file:
            text = file.read()

        text = text.replace("\n\n", " <eos> ")
        self.tokens = tokenizer.encode(text).ids

        if len(self.tokens) < 2:
            raise ValueError(f"Dataset must contain at least two tokens: {file_path}")
        
        self.seq_len = seq_len

    def __len__(self):
        return (len(self.tokens) - 2) // self.seq_len + 1

    def __getitem__(self, idx):

        start = idx * self.seq_len
        end = start + self.seq_len + 1
        chunk = self.tokens[start:end]

        if len(chunk) < self.seq_len + 1:
            chunk += [0] * (self.seq_len + 1 - len(chunk))

        return (torch.tensor(chunk[:-1], dtype=torch.long),
                torch.tensor(chunk[1:], dtype=torch.long))

class MixedDataset(Dataset):

    def __init__(self, datasets, weights=None):

        if not datasets:
            raise ValueError("At least one training dataset is required")
        
        self.datasets = datasets

        if weights is None:
            weights = [1.0 / len(datasets)] * len(datasets)

        if len(weights) != len(datasets) or sum(weights) <= 0:
            raise ValueError("Dataset weights must match the datasets and sum to a positive value")
        
        total_weight = sum(weights)

        self.cumulative_weights = np.cumsum([weight / total_weight for weight in weights])
        self.total_length = sum(len(dataset) for dataset in datasets)

    def __len__(self):
        return self.total_length

    def __getitem__(self, idx):

        dataset_idx = min(np.searchsorted(self.cumulative_weights, random.random()), len(self.datasets) - 1)
        dataset = self.datasets[dataset_idx]

        return dataset[random.randrange(len(dataset))]

# Training the Tokeniser

In [ ]:
def train_tokenizer(stage_name, vocab_size):

    tokenizer_path = TOKENIZER_SAVE_DIR / f"tokenizer_{vocab_size}_literature_v2.json"

    if tokenizer_path.exists():

        print(f"Loading existing tokenizer from {tokenizer_path}")
        return Tokenizer.from_file(str(tokenizer_path))

    all_texts = []

    for stage in STAGE_ORDER:
        for key, path in STAGE_DATA_CONFIG[stage].items():

            if key in ("main", "dialogues", "narrative") and Path(path).is_file():
                text = Path(path).read_text(encoding="utf-8")

                if key == "dialogues" and ("<user>" in text or "<assistant>" in text):
                    raise ValueError(f"Stale role-labeled dialogue data at {path}; regenerate it with Scripts/data_preprocessing.py before training.")
                
                all_texts.append(text)

    if not all_texts:
        raise FileNotFoundError("No training text found. Run Scripts/data_preprocessing.py first.")

    special_tokens = ["<pad>", "<unk>", "<bos>", "<eos>", "<user>", "<assistant>"]

    tokenizer = Tokenizer(models.BPE())
    tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
    
    trainer = trainers.BpeTrainer(vocab_size=vocab_size,
                                  special_tokens=special_tokens,
                                  
                                  show_progress=True,
                                  min_frequency=2)
    
    tokenizer.train_from_iterator(all_texts, trainer=trainer)
    tokenizer.save(str(tokenizer_path))

    print(f"Tokenizer saved to {tokenizer_path}")
    return tokenizer

# Building the LLM Model

In [ ]:
class SimpleTransformer(nn.Module):

    def __init__(self, vocab_size, embed_dim, num_heads, hidden_dim, num_layers, seq_len):

        super(SimpleTransformer, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = nn.Embedding(seq_len, embed_dim)
        self.dropout = nn.Dropout(0.1)

        self.transformer_blocks = nn.ModuleList([
                                    nn.TransformerEncoderLayer(d_model=embed_dim,
                                                               nhead=num_heads,
                                                               
                                                               dim_feedforward=hidden_dim,
                                                               dropout=0.1,
                                                               batch_first=False) for _ in range(num_layers)])
        
        self.norm = nn.LayerNorm(embed_dim)
        self.fc = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):

        positions = torch.arange(0, x.size(1), device=x.device).unsqueeze(0)

        x = self.embedding(x) + self.position_embedding(positions)
        x = self.dropout(x)
        x = x.transpose(0, 1) 

        mask = torch.triu(torch.ones(x.size(0), x.size(0), device=x.device) * float('-inf'), diagonal=1)

        for block in self.transformer_blocks:
            x = block(x, src_mask=mask)

        x = x.transpose(0, 1)
        x = self.norm(x)
        
        return self.fc(x)

# Saving the Model

In [ ]:
def delete_old_best_model(model_dir, stage_name):

    for filename in os.listdir(model_dir):

        if filename.startswith(f"{stage_name}_best"):
            os.remove(os.path.join(model_dir, filename))

def save_best_model(model, model_save_dir, settings, stage_name, epoch, validation_loss):

    model_filename = f"{stage_name}_best.pth"
    settings_filename = f"{stage_name}_best_settings.json"
    model_path = os.path.join(model_save_dir, model_filename)
    settings_path = os.path.join(model_save_dir, settings_filename)

    settings_with_metadata = settings.copy()
    settings_with_metadata["best_epoch"] = epoch
    settings_with_metadata["best_validation_loss"] = float(validation_loss)

    torch.save(model.state_dict(), model_path)

    with open(settings_path, "w", encoding="utf-8") as file:
        json.dump(settings_with_metadata, file, indent=2)

    logging.info("%s best checkpoint saved at epoch %d (validation loss %.4f)", stage_name, epoch, validation_loss,)
    return model_path

# Training the Model

In [ ]:
def train_on_stage(stage_name, stage_config, model_settings, tokenizer, previous_model_path=None):

    print(f"\n{'=' * 60}\nTraining Stage: {stage_name.upper()}\n{'=' * 60}")

    main_path = Path(stage_config["main"])
    validation_path = Path(stage_config["validation"])

    for required_path in (main_path, validation_path):
        
        if not required_path.is_file():
            raise FileNotFoundError(f"Required {stage_name} data missing: {required_path}. Run Scripts/data_preprocessing.py from the project root first.")

    datasets_to_use = [TextDataset(str(main_path), tokenizer)]
    weights = [0.5]

    for key, enabled, weight in (("dialogues", "use_dialogues", 0.3), ("narrative", "use_narrative", 0.05)):

        if stage_config.get(enabled):
            optional_path = Path(stage_config[key])
            
            if optional_path.is_file():
                
                if key == "dialogues":
                    dialogue_text = optional_path.read_text(encoding="utf-8")
                    
                    if "<user>" in dialogue_text or "<assistant>" in dialogue_text:
                        raise ValueError(f"Stale role-labeled dialogue data at {optional_path}; regenerate it with Scripts/data_preprocessing.py.")
                    
                datasets_to_use.append(TextDataset(str(optional_path), tokenizer))
                weights.append(weight)
            
            else:
                logging.warning("Optional %s source missing for %s: %s", key, stage_name, optional_path)

    training_data = (MixedDataset(datasets_to_use, weights) if len(datasets_to_use) > 1 else datasets_to_use[0])
    training_loader = DataLoader(training_data, batch_size=BATCH_SIZE, shuffle=True)

    validation_data = TextDataset(str(validation_path), tokenizer)
    validation_loader = DataLoader(validation_data, batch_size=BATCH_SIZE, shuffle=False)
    
    print(f"Training chunks: {len(training_data)} | Held-out chunks: {len(validation_data)}")

    model = SimpleTransformer(vocab_size=VOCAB_SIZE,
                              embed_dim=EMBED_DIM,
                              num_heads=NUM_HEADS,

                              hidden_dim=HIDDEN_DIM,
                              num_layers=NUM_LAYERS,
                              seq_len=SEQ_LEN).to(DEVICE)

    if previous_model_path is not None:

        if not os.path.isfile(previous_model_path):
            raise FileNotFoundError(f"Previous stage checkpoint not found: {previous_model_path}")
        
        model.load_state_dict(torch.load(previous_model_path, map_location=DEVICE, weights_only=True))
        current_lr = FINE_TUNE_LR

        print(f"Transfer learning enabled (LR: {current_lr})")
    
    else:
        current_lr = LEARNING_RATE
        print(f"Training from scratch (LR: {current_lr})")

    loss_fn = nn.CrossEntropyLoss(ignore_index=0)
    optimizer = AdamW(model.parameters(), lr=current_lr)

    scheduler = get_scheduler("linear",
                              optimizer=optimizer,
                              num_warmup_steps=WARMUP_STEPS,
                              num_training_steps=len(training_loader) * EPOCHS)
    
    best_validation_loss = float("inf")

    patience_count = 0
    best_model_path = None

    def evaluate():

        model.eval()
        total_loss = 0.0
        total_tokens = 0

        with torch.no_grad():

            for input_ids, target_ids in validation_loader:

                input_ids = input_ids.to(DEVICE)
                target_ids = target_ids.to(DEVICE)
                
                outputs = model(input_ids)
                
                loss = loss_fn(outputs.reshape(-1, VOCAB_SIZE), target_ids.reshape(-1))
                valid_tokens = int((target_ids != 0).sum().item())
                
                total_loss += loss.item() * valid_tokens
                
                total_tokens += valid_tokens
        if total_tokens == 0:
            
            raise ValueError(f"Validation set for {stage_name} contains no non-padding targets")
        return total_loss / total_tokens

    for epoch in range(EPOCHS):
        
        model.train()
        training_loss = 0.0
        
        for input_ids, target_ids in tqdm(training_loader, leave=True, desc=f"[{stage_name}] Epoch {epoch + 1}"):
            
            input_ids = input_ids.to(DEVICE)
            target_ids = target_ids.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(input_ids)

            loss = loss_fn(outputs.reshape(-1, VOCAB_SIZE), target_ids.reshape(-1))
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            
            optimizer.step()
            scheduler.step()
            
            training_loss += loss.item()

        average_training_loss = training_loss / len(training_loader)
        validation_loss = evaluate()
        
        print(f"Epoch {epoch + 1:3d} | Train: {average_training_loss:.4f} "
              f"| Validation: {validation_loss:.4f} | Best validation: {best_validation_loss:.4f}")
        
        logging.info("%s - Epoch %d - Train loss: %.4f - Validation loss: %.4f", stage_name, epoch + 1, average_training_loss, validation_loss)

        if validation_loss < best_validation_loss:

            best_validation_loss = validation_loss
            patience_count = 0
            
            best_model_path = save_best_model(model, str(MODEL_SAVE_DIR), model_settings, stage_name, epoch + 1, validation_loss)
            print("New best validation checkpoint saved.")

        else:
            patience_count += 1
            print(f"Validation patience: {patience_count}/{VALIDATION_PATIENCE}")
            
            if patience_count >= VALIDATION_PATIENCE:
                print("Early stopping: validation loss did not improve.")
                break

    return best_model_path

# Start

In [ ]:
print("\n" + "=" * 60)
print("TRAINING CONVERSATIONAL LANGUAGE MODEL")
print("=" * 60)

model_settings = {"VOCAB_SIZE": VOCAB_SIZE,
                  "EMBED_DIM": EMBED_DIM,

                  "NUM_LAYERS": NUM_LAYERS,
                  "NUM_HEADS": NUM_HEADS,

                  "HIDDEN_DIM": HIDDEN_DIM,
                  "BATCH_SIZE": BATCH_SIZE,
                  "SEQ_LEN": SEQ_LEN,
                  "EPOCHS": EPOCHS,

                  "LEARNING_RATE": LEARNING_RATE,
                  "TOKENIZER_PATH": TOKENIZER_RELATIVE_PATH}

tokenizer = train_tokenizer("all_stages", VOCAB_SIZE)
previous_model_path = None

for stage_name in STAGE_ORDER:

    previous_model_path = train_on_stage(stage_name=stage_name,
                                         stage_config=STAGE_DATA_CONFIG[stage_name],

                                         model_settings=model_settings,
                                         tokenizer=tokenizer,
                                         
                                         previous_model_path=previous_model_path)

    if previous_model_path is None:
        raise RuntimeError(f"Stage {stage_name} did not produce a best validation checkpoint")

print(f"\nTraining complete. Final model: {previous_model_path}")